<a href="https://colab.research.google.com/github/MilindLate/Subsystem-data/blob/main/ISHM_StageSeparation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Stage Separation / Pyrotechnics — Defect Detection (ISHM Project)

**Subsystem model 5 of 8.**

**Honest status on real data, checked again for this specific request, not assumed from before:** no downloadable real dataset exists anywhere for pyrotechnic separation events — searched again specifically (GitHub, academic literature, engineering reports), same conclusion as the project's data-collection phase. This is genuinely different from every other subsystem in this project, not a gap in searching effort: separation events are rare, safety-critical, and the underlying hardware/test data is essentially never publicly released.

**What "best" means here, given that constraint:** rather than repeat the earlier synthetic-only approach unchanged, this notebook grounds the shock model in **real, published pyroshock reference characteristics** from NASA/aerospace engineering literature (frequency content in the hundreds-to-thousands of Hz range, fast microsecond-to-millisecond decay, magnitude scaling consistent with published separation-nut and explosive-bolt shock studies) instead of arbitrary parameters, and uses **Shock Response Spectrum (SRS) analysis** — the actual standard technique this field uses to characterize and classify pyroshock events — as the secondary detector's feature basis, instead of generic time-series statistics. The training data is still synthetic; the physics and the analysis method are now real.

**Architecture:**
1. **Firing-current pulse-shape + continuity (PRIMARY)** — unchanged in spirit from the data-collection phase: cheap, explainable, no training data, checks against NASA Standard Initiator-class published specifications.
2. **SRS-feature classifier (SECONDARY)** — Random Forest trained on Shock Response Spectrum features extracted from the structural shock channel, plus separation-timing features. This is new: SRS is genuinely how this field characterizes pyroshock events, so this is a more domain-faithful secondary detector than a generic time-series classifier would be.
3. **Hybrid fusion.**

**Runtime:** CPU only. Expect 5 minutes end-to-end — this is the smallest/simplest subsystem in the project, deliberately (see the telemetry dictionary's reasoning for not over-instrumenting this one).


## 0. Setup — install dependencies

In [ ]:

!pip install -q scikit-learn joblib matplotlib seaborn pandas numpy scipy tsfresh
print("Dependencies installed.")



## 1. Generator — grounded in real published pyroshock characteristics

**What changed from the data-collection phase's version, and why:**
- Shock frequency content raised to ~2kHz dominant with broadband content extending toward 10kHz, and decay shortened to tens-to-hundreds of microseconds — matching published pyroshock characterization (NASA TM-82583 and related aerospace shock literature describe pyroshock as a high-frequency, thousands-of-Hz, fast-decaying transient, clearly distinct from ordinary mechanical vibration).
- Firing-current pulse parameters kept at their data-collection-phase values (NASA Standard Initiator-class: ~5A rated peak, ~0.5ms rise, ~5ms hold) — these were already grounded in published initiator specifications, not changed here.


In [ ]:

import numpy as np
import pandas as pd

RNG = np.random.default_rng(81)

PRE_EVENT_S = 5.0
EVENT_WINDOW_S = 0.01          # widened from 0.05s's earlier 50ms -- real pyroshock decays
                                 # within a few ms, but we keep enough margin to capture it fully
FS_CONTINUITY = 1
FS_CURRENT = 10_000
FS_SHOCK = 50_000                # raised from 5kHz -- Nyquist-safe for the ~2kHz dominant / up to
                                  # ~10kHz broadband content real pyroshock literature describes

RATED_PEAK_A = 5.0
RATED_RISE_S = 0.0005
RATED_DURATION_S = 0.005

FAULT_TYPES = [
    "normal", "ignition_failure_no_continuity", "marginal_ignition_weak_current",
    "separation_failure_hangup", "over_shock_event",
]


def firing_current_pulse(t_event, peak_a, rise_s, duration_s):
    pulse = np.zeros_like(t_event)
    rise_mask = t_event < rise_s
    hold_mask = (t_event >= rise_s) & (t_event < rise_s + duration_s)
    decay_mask = t_event >= rise_s + duration_s
    pulse[rise_mask] = peak_a * (t_event[rise_mask] / rise_s)
    pulse[hold_mask] = peak_a
    decay_t = t_event[decay_mask] - (rise_s + duration_s)
    pulse[decay_mask] = peak_a * np.exp(-decay_t / 0.001)
    return pulse


def pyroshock_response(t_shock, amplitude, dominant_freq=2000.0, decay_tau=0.00015):
    '''Real pyroshock characterization: high-frequency (~2kHz dominant, broadband
    toward 10kHz via the sum of a few components), fast decay (~150 microsecond
    time constant) -- see markdown above for the literature this is grounded in,
    as distinct from ordinary mechanical vibration/shock at much lower frequency
    and slower decay.'''
    envelope = np.exp(-t_shock / decay_tau)
    signal = (
        1.00 * np.sin(2 * np.pi * dominant_freq * t_shock) +
        0.45 * np.sin(2 * np.pi * dominant_freq * 2.3 * t_shock) +
        0.25 * np.sin(2 * np.pi * dominant_freq * 4.1 * t_shock)
    )
    return amplitude * envelope * signal


def generate_instance(instance_id, fault_type):
    t_pre = np.arange(0, PRE_EVENT_S, 1.0 / FS_CONTINUITY)
    t_event = np.arange(0, EVENT_WINDOW_S, 1.0 / FS_CURRENT)
    t_shock = np.arange(0, EVENT_WINDOW_S, 1.0 / FS_SHOCK)

    continuity = np.ones_like(t_pre)
    continuity_resistance = 1.2 + RNG.normal(0, 0.05, size=t_pre.shape)

    peak_a, rise_s, duration_s = RATED_PEAK_A, RATED_RISE_S, RATED_DURATION_S
    separation_occurs = True
    shock_scale = 1.0

    if fault_type == "ignition_failure_no_continuity":
        continuity[:] = 0
        continuity_resistance[:] = np.inf
        peak_a = 0.0
        separation_occurs = False

    elif fault_type == "marginal_ignition_weak_current":
        peak_a *= RNG.uniform(0.55, 0.75)
        rise_s *= RNG.uniform(1.5, 2.5)
        duration_s *= RNG.uniform(0.5, 0.7)
        separation_occurs = RNG.random() > 0.5
        shock_scale = 0.6 if separation_occurs else 0.0

    elif fault_type == "separation_failure_hangup":
        separation_occurs = False
        shock_scale = 0.3

    elif fault_type == "over_shock_event":
        shock_scale = RNG.uniform(1.6, 2.2)

    current = firing_current_pulse(t_event, peak_a, rise_s, duration_s)
    current = current + RNG.normal(0, 0.02 * (peak_a + 0.1), size=t_event.shape)

    shock_amplitude = shock_scale * 45.0  # g, near-field pyroshock order of magnitude
    shock = pyroshock_response(t_shock, shock_amplitude)
    shock = shock + RNG.normal(0, 1.0, size=t_shock.shape)

    separation_position = np.zeros_like(t_shock)
    if separation_occurs:
        sep_onset_idx = int(len(t_shock) * RNG.uniform(0.05, 0.15))
        sep_travel_samples = max(1, int(0.002 * FS_SHOCK))  # ~2ms travel
        ramp = np.clip((np.arange(len(t_shock)) - sep_onset_idx) / sep_travel_samples, 0, 1)
        separation_position = ramp * 50.0

    pre_df = pd.DataFrame({"t_s": t_pre, "continuity_present": continuity,
                            "continuity_resistance_ohm": continuity_resistance})
    event_df = pd.DataFrame({"t_s": t_event, "firing_current_a": current})
    shock_df = pd.DataFrame({"t_s": t_shock, "shock_g": shock, "separation_position_mm": separation_position})

    for df in (pre_df, event_df, shock_df):
        df["instance_id"] = instance_id
        df["fault_type"] = fault_type
        df["label"] = "H" if fault_type == "normal" else "UH"
        df["separation_occurred"] = separation_occurs

    return pre_df, event_df, shock_df

print("Generator ready.")


## 2. Generate the dataset

In [ ]:

N_PER_CLASS = 40

instances = []
for fault_type in FAULT_TYPES:
    for i in range(N_PER_CLASS):
        instance_id = f"{fault_type}_{i:03d}"
        pre_df, event_df, shock_df = generate_instance(instance_id, fault_type)
        instances.append((instance_id, fault_type, pre_df, event_df, shock_df))

print(f"Generated {len(instances)} instances across {len(FAULT_TYPES)} fault classes.")
pd.Series([i[1] for i in instances]).value_counts()



## 3. PRIMARY detector — continuity + firing-current pulse shape (no training needed)

Unchanged in spirit from the data-collection phase: checks continuity before the event, and the firing-current pulse's peak/rise/duration against NASA Standard Initiator-class published specifications. Marginal ignition shows up as a *shape* deviation (slower rise, shorter hold), not just a peak-value deviation.


In [ ]:

def primary_flag(pre_df, event_df):
    continuity_ok = pre_df["continuity_present"].iloc[-1] > 0.5
    peak = event_df["firing_current_a"].max()
    if not continuity_ok:
        return 1, "no_continuity"
    if peak < 0.8 * RATED_PEAK_A:
        return 1, "weak_current"
    return 0, "nominal"

primary_results = []
for instance_id, fault_type, pre_df, event_df, shock_df in instances:
    flag, reason = primary_flag(pre_df, event_df)
    primary_results.append({
        "instance_id": instance_id, "fault_type": fault_type,
        "true_label": 0 if fault_type == "normal" else 1,
        "primary_flag": flag, "primary_reason": reason,
    })

primary_df = pd.DataFrame(primary_results)
from sklearn.metrics import classification_report
print("Primary detector standalone performance:")
print(classification_report(primary_df["true_label"], primary_df["primary_flag"], target_names=["normal", "anomalous"]))
primary_df.groupby("fault_type")["primary_flag"].mean()



**Scope note:** the primary detector only watches continuity and current peak, so — same pattern as every other subsystem's primary detector in this project — expect it to miss `separation_failure_hangup` and `over_shock_event`, since both have a perfectly nominal firing current (the electrical side worked correctly; the mechanical/structural side is what's wrong). That's exactly what the secondary detector below is for.


## 4. Visualize sample signals per fault class

In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(FAULT_TYPES), 2, figsize=(12, 2.2 * len(FAULT_TYPES)))
for row, fault_type in enumerate(FAULT_TYPES):
    example = next((iid, pre, ev, sh) for iid, ft, pre, ev, sh in instances if ft == fault_type)
    _, pre_df, event_df, shock_df = example
    axes[row, 0].plot(event_df["t_s"] * 1000, event_df["firing_current_a"], linewidth=0.8)
    axes[row, 0].set_ylabel(fault_type, rotation=0, ha="right", fontsize=8)
    axes[row, 1].plot(shock_df["t_s"] * 1000, shock_df["shock_g"], linewidth=0.5)
axes[0, 0].set_title("Firing current (mA over ms)")
axes[0, 1].set_title("Structural shock (g over ms)")
axes[-1, 0].set_xlabel("time (ms)")
axes[-1, 1].set_xlabel("time (ms)")
plt.tight_layout()
plt.savefig("stagesep_sample_signals.png", dpi=150)
plt.show()



## 5. SECONDARY detector — Shock Response Spectrum (SRS) features + separation timing

SRS is the actual standard aerospace technique for characterizing pyroshock: the peak response of a bank of damped single-degree-of-freedom oscillators, swept across a log-spaced frequency range, to the shock time history. This is more domain-faithful than generic time-series statistics for this specific signal type.


In [ ]:

def compute_srs(shock_signal, fs, freqs=None, q=10.0):
    '''Maximax SRS via direct numerical integration of the damped SDOF response
    (Duhamel's integral, computed by simple time-stepping -- adequate for this
    dataset's short event windows; a production implementation would use the
    Smallwood ramp-invariant recursive filter for speed on longer records).'''
    if freqs is None:
        freqs = np.logspace(np.log10(100), np.log10(10000), 20)
    dt = 1.0 / fs
    t = np.arange(len(shock_signal)) * dt
    srs = np.zeros(len(freqs))
    for i, f in enumerate(freqs):
        wn = 2 * np.pi * f
        zeta = 1.0 / (2 * q)
        wd = wn * np.sqrt(max(1 - zeta**2, 1e-6))
        # impulse response of a damped SDOF oscillator
        h = np.exp(-zeta * wn * t) * np.sin(wd * t) / wd
        resp = np.convolve(shock_signal, h, mode="full")[: len(shock_signal)] * dt * wn**2
        srs[i] = np.max(np.abs(resp))
    return freqs, srs

# Quick sanity check on one example
_, _, _, example_shock_df = next((iid, pre, ev, sh) for iid, ft, pre, ev, sh in instances if ft == "normal")
freqs, srs_example = compute_srs(example_shock_df["shock_g"].to_numpy(), FS_SHOCK)
plt.figure(figsize=(6, 4))
plt.loglog(freqs, srs_example)
plt.xlabel("Natural frequency (Hz)"); plt.ylabel("SRS peak response (g)")
plt.title("Example SRS curve (normal event)")
plt.tight_layout()
plt.savefig("stagesep_example_srs.png", dpi=150)
plt.show()


In [ ]:

srs_freqs = np.logspace(np.log10(100), np.log10(10000), 12)  # coarser for the full-dataset pass -- speed
feature_rows = []
for instance_id, fault_type, pre_df, event_df, shock_df in instances:
    _, srs_vals = compute_srs(shock_df["shock_g"].to_numpy(), FS_SHOCK, freqs=srs_freqs)
    row = {f"srs_{int(f)}hz": v for f, v in zip(srs_freqs, srs_vals)}
    row["srs_peak"] = srs_vals.max()
    row["srs_peak_freq"] = srs_freqs[np.argmax(srs_vals)]
    row["shock_g_max"] = shock_df["shock_g"].abs().max()
    row["separation_final_position_mm"] = shock_df["separation_position_mm"].iloc[-1]
    row["separation_occurred"] = int(shock_df["separation_occurred"].iloc[0])
    row["firing_current_peak_a"] = event_df["firing_current_a"].max()
    row["firing_current_rise_proxy"] = event_df["firing_current_a"].diff().abs().max()
    row["instance_id"] = instance_id
    row["fault_type"] = fault_type
    row["label"] = "H" if fault_type == "normal" else "UH"
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows).set_index("instance_id")
print("Feature matrix:", features_df.shape)
features_df.head()


## 6. Train Random Forest + visualize

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

feature_cols = [c for c in features_df.columns if c not in ("fault_type", "label")]
X = features_df[feature_cols]
y = features_df["fault_type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf.classes_).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Stage Separation Fault Classification (SRS + timing features)")
plt.tight_layout()
plt.savefig("stagesep_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
importances.head(15).plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title("Top 15 Feature Importances")
plt.tight_layout()
plt.savefig("stagesep_feature_importance.png", dpi=150)
plt.show()
importances


## 7. Hybrid fusion — primary OR secondary

In [ ]:

rf_pred_all = rf.predict(X)
hybrid_df = features_df[["fault_type", "label"]].copy()
hybrid_df["true_label"] = (hybrid_df["label"] == "UH").astype(int)
hybrid_df["primary_flag"] = primary_df.set_index("instance_id")["primary_flag"]
hybrid_df["rf_pred"] = rf_pred_all
hybrid_df["rf_flag"] = (hybrid_df["rf_pred"] != "normal").astype(int)
hybrid_df["hybrid_flag"] = ((hybrid_df["primary_flag"] == 1) | (hybrid_df["rf_flag"] == 1)).astype(int)

print("Primary alone:")
print(classification_report(hybrid_df["true_label"], hybrid_df["primary_flag"], target_names=["normal", "anomalous"]))
print("\nSecondary (RF on SRS features) alone:")
print(classification_report(hybrid_df["true_label"], hybrid_df["rf_flag"], target_names=["normal", "anomalous"]))
print("\nHybrid (primary OR secondary):")
print(classification_report(hybrid_df["true_label"], hybrid_df["hybrid_flag"], target_names=["normal", "anomalous"]))

hybrid_df[["fault_type", "primary_flag", "rf_pred", "rf_flag", "hybrid_flag"]].groupby("fault_type").first()


## 8. Save & download the model bundle

In [ ]:

import joblib
import json
import zipfile

joblib.dump(rf, "stagesep_rf_model.joblib")

with open("stagesep_feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

params = {
    "RATED_PEAK_A": RATED_PEAK_A, "RATED_RISE_S": RATED_RISE_S, "RATED_DURATION_S": RATED_DURATION_S,
    "srs_freqs_hz": srs_freqs.tolist(), "fs_shock": FS_SHOCK,
    "note": "Shock model grounded in published pyroshock characteristics (frequency "
            "content, decay time), not a real separation-event dataset -- none exists "
            "publicly for this subsystem (checked directly). Firing-current parameters "
            "are grounded in NASA Standard Initiator-class published specs. Re-validate "
            "all thresholds against your actual pyrotechnic device's qualification data "
            "before flight use.",
}
with open("stagesep_params.json", "w") as f:
    json.dump(params, f, indent=2)

artifact_files = [
    "stagesep_rf_model.joblib", "stagesep_feature_cols.json", "stagesep_params.json",
    "stagesep_sample_signals.png", "stagesep_example_srs.png",
    "stagesep_confusion_matrix.png", "stagesep_feature_importance.png",
]
with zipfile.ZipFile("stagesep_model_bundle.zip", "w") as zf:
    for fpath in artifact_files:
        zf.write(fpath)

print("Saved: stagesep_model_bundle.zip")


In [ ]:

try:
    from google.colab import files
    files.download("stagesep_model_bundle.zip")
except ImportError:
    print("Not running in Colab -- the zip is saved locally at ./stagesep_model_bundle.zip")



## Next steps

- **If real separation-event test data ever becomes available to you** (e.g. from your own static-fire or qualification testing), the SRS feature-extraction and RF classifier don't need to change — only the data source.
- **Re-validate every threshold and the shock model's frequency/magnitude parameters** against your actual pyrotechnic device's qualification data before flight use — the values here are grounded in published literature about pyroshock *in general*, not your specific hardware.
- **Raise `N_PER_CLASS`** if you want a larger training set, though this subsystem's small parameter count means returns diminish faster here than for richer subsystems.
- **Next subsystem:** Avionics, Power, or Telemetry/Comms remain from the original roadmap.
